# Notebook 16 -- Advanced Topics: Hamiltonian Simulation, HHL, and Beyond

**Prerequisites:** Notebooks 01-08 and 11. Familiarity with QPE, Pauli operators, and noise.

This capstone notebook surveys the advanced algorithmic features of the
Goqu SDK. We bring together concepts from across the curriculum to explore
**Hamiltonian simulation**, the **HHL algorithm** for linear systems,
**Clifford simulation** at massive scale, and **Pauli algebra**.

By the end of this notebook you will be able to:

1. **Implement** Trotter-Suzuki Hamiltonian simulation and compare first vs second order.
2. **Describe** the HHL algorithm for solving linear systems.
3. **Demonstrate** Clifford simulation scaling to thousands of qubits.
4. **Use** Pauli algebra to multiply, commute, and compute expectation values.

| Topic | Key idea |
|-------|----------|
| Trotter-Suzuki | Approximate $e^{-iHt}$ by slicing into small steps |
| HHL | Solve $Ax = b$ on a quantum computer |
| Clifford simulation | Stabilizer states up to 100,000 qubits |
| Pauli algebra | Multiply, commute, and compute expectation values |

For pulse programming, operator representations, QASM/Quil interop, backends, and observability, see [Notebook 16b](16b-sdk-reference.ipynb).

In [ ]:
import (
	"context"
	"fmt"

	"github.com/splch/goqu/algorithm/hhl"
	"github.com/splch/goqu/algorithm/trotter"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/sim/clifford"
	"github.com/splch/goqu/sim/pauli"
	"github.com/splch/goqu/sim/statevector"
)

## Trotter-Suzuki Hamiltonian Simulation

Many quantum algorithms require simulating the time evolution of a
physical system whose dynamics are governed by a Hamiltonian $H$. The
time-evolution operator is

$$U(t) = e^{-iHt}$$

When $H$ is a sum of non-commuting terms $H = \sum_k H_k$, the matrix
exponential $e^{-i(H_1 + H_2 + \cdots)t}$ cannot be factored exactly.
The **Trotter-Suzuki decomposition** approximates it by splitting into
small time steps:

- **First-order (Lie-Trotter):**
  $e^{-iHt} \approx \left[\prod_k e^{-iH_k \Delta t}\right]^n$
  where $\Delta t = t/n$.

- **Second-order (Suzuki-Trotter):**
  $e^{-iHt} \approx \left[\prod_k e^{-iH_k \Delta t/2} \cdot \prod_{k'} e^{-iH_{k'} \Delta t/2}\right]^n$
  where the second product runs in reverse order. This symmetric
  splitting cancels the leading-order error term.

Let's simulate time evolution of a simple 2-qubit Heisenberg XX+ZZ
Hamiltonian: $H = X \otimes X + Z \otimes Z$.

In [ ]:
%%
ctx := context.Background()

// H = X*X + Z*Z (simple 2-qubit Hamiltonian)
xx, _ := pauli.Parse("XX")
zz, _ := pauli.Parse("ZZ")
ham, _ := pauli.NewPauliSum([]pauli.PauliString{xx, zz})

result, _ := trotter.Run(ctx, trotter.Config{
	Hamiltonian: ham,
	Time:        1.0,
	Steps:       4,
	Order:       trotter.First,
})

fmt.Printf("Trotter circuit: %d gates, depth %d\n",
	result.Circuit.Stats().GateCount, result.Circuit.Stats().Depth)
fmt.Println(draw.String(result.Circuit))

In [ ]:
%%
// Compare first-order vs second-order Trotter decompositions.
// More Trotter steps and higher order both improve accuracy at the
// cost of deeper circuits.
ctx := context.Background()
xx, _ := pauli.Parse("XX")
zz, _ := pauli.Parse("ZZ")
ham, _ := pauli.NewPauliSum([]pauli.PauliString{xx, zz})

r1, _ := trotter.Run(ctx, trotter.Config{
	Hamiltonian: ham, Time: 1.0, Steps: 4, Order: trotter.First,
})
r2, _ := trotter.Run(ctx, trotter.Config{
	Hamiltonian: ham, Time: 1.0, Steps: 4, Order: trotter.Second,
})

fmt.Printf("First-order:  %d gates, depth %d\n",
	r1.Circuit.Stats().GateCount, r1.Circuit.Stats().Depth)
fmt.Printf("Second-order: %d gates, depth %d\n",
	r2.Circuit.Stats().GateCount, r2.Circuit.Stats().Depth)

// Use Evolve to get final statevectors and compare fidelity.
sv1, _ := trotter.Evolve(ctx, trotter.Config{
	Hamiltonian: ham, Time: 1.0, Steps: 4, Order: trotter.First,
}, nil)
sv2, _ := trotter.Evolve(ctx, trotter.Config{
	Hamiltonian: ham, Time: 1.0, Steps: 4, Order: trotter.Second,
}, nil)

// High-precision reference (many steps).
svRef, _ := trotter.Evolve(ctx, trotter.Config{
	Hamiltonian: ham, Time: 1.0, Steps: 100, Order: trotter.Second,
}, nil)

// Compute fidelity |<ref|approx>|^2.
fidelity := func(a, b []complex128) float64 {
	var dot complex128
	for i := range a {
		dot += complex(real(a[i]), -imag(a[i])) * b[i]
	}
	return real(dot)*real(dot) + imag(dot)*imag(dot)
}

fmt.Printf("\nFidelity vs high-precision reference:\n")
fmt.Printf("  First-order (4 steps):  %.8f\n", fidelity(svRef, sv1))
fmt.Printf("  Second-order (4 steps): %.8f\n", fidelity(svRef, sv2))
fmt.Println("\nSecond-order Trotter is more accurate at the same step count.")

## HHL Algorithm for Linear Systems

The **Harrow-Hassidim-Lloyd (HHL)** algorithm solves the linear system
$Ax = b$ by encoding the solution in the amplitudes of a quantum state.
For an $N \times N$ Hermitian matrix $A$ with eigenvalues $\lambda_j$
and eigenvectors $|u_j\rangle$, if we decompose $|b\rangle = \sum_j
\beta_j |u_j\rangle$, then the solution is $|x\rangle \propto \sum_j
(\beta_j / \lambda_j) |u_j\rangle$.

The algorithm uses three quantum registers:

1. **Ancilla** (1 qubit) -- flags successful eigenvalue inversion.
2. **Phase register** -- stores eigenvalue estimates via QPE.
3. **System register** -- encodes $|b\rangle$ and ultimately $|x\rangle$.

The success probability indicates how often post-selection succeeds.

In [ ]:
%%
ctx := context.Background()

// Simple 2x2 system: A = 0.5*Z (diagonal Hamiltonian).
// Eigenvalues are +0.5 and -0.5.
// |b> = H|0> = |+> = (|0> + |1>) / sqrt(2)
zi, _ := pauli.Parse("Z")
A, _ := pauli.NewPauliSum([]pauli.PauliString{zi.Scale(0.5)})
rhs, _ := builder.New("rhs", 1).H(0).Build()

result, err := hhl.Run(ctx, hhl.Config{
	Matrix:       A,
	RHS:          rhs,
	NumPhaseBits: 3,
	NumQubits:    1,
	Shots:        1000,
})
if err != nil {
	fmt.Println("Error:", err)
} else {
	fmt.Printf("HHL success probability: %.4f\n", result.Success)
	fmt.Printf("Solution state vector:   ")
	for i, amp := range result.StateVector {
		fmt.Printf("|%d>: %.4f+%.4fi  ", i, real(amp), imag(amp))
	}
	fmt.Println()
	fmt.Printf("\nHHL circuit: %d qubits, %d gates, depth %d\n",
		result.Circuit.NumQubits(),
		result.Circuit.Stats().GateCount,
		result.Circuit.Stats().Depth)
}

## Clifford Simulation -- Stabilizer States at Scale

Clifford circuits (composed of H, S, CNOT, and Pauli gates) can be
simulated efficiently on classical hardware using the **stabilizer
tableau** formalism. Where a full statevector simulator needs $2^n$
amplitudes, the Clifford simulator only needs $O(n^2)$ bits.

This makes it possible to simulate circuits with **thousands or even
100,000 qubits** -- far beyond any statevector simulator. The catch:
only Clifford gates are allowed (no T gate, no arbitrary rotations).

Let's create a 1000-qubit GHZ state: $|\text{GHZ}\rangle = (|0\rangle^{\otimes 1000} + |1\rangle^{\otimes 1000}) / \sqrt{2}$.

In [ ]:
%%
nQubits := 1000
b := builder.New("ghz-1000", nQubits)
b.H(0)
for i := 0; i < nQubits-1; i++ {
	b.CNOT(i, i+1)
}
b.MeasureAll()
c, err := b.Build()
if err != nil {
	fmt.Println("Build error:", err)
}

sim := clifford.New(nQubits)
counts, err := sim.Run(c, 100)
if err != nil {
	fmt.Println("Run error:", err)
}

// Should see only all-0s and all-1s.
fmt.Printf("1000-qubit GHZ state -- %d distinct outcomes from 100 shots:\n", len(counts))
for state, n := range counts {
	fmt.Printf("  |%s...%s> (%d qubits): %d shots\n",
		state[:3], state[len(state)-3:], len(state), n)
}
fmt.Println("\nA statevector simulator would need 2^1000 amplitudes for this.")
fmt.Println("The Clifford simulator handles it in milliseconds with O(n^2) memory.")

## Pauli Algebra

The Pauli operators $\{I, X, Y, Z\}$ form a basis for all $2 \times 2$
Hermitian matrices. Multi-qubit Pauli **strings** (tensor products like
$X \otimes Z \otimes I$) and their linear combinations (**Pauli sums**)
are the natural language for expressing Hamiltonians and observables.

Key algebraic properties:

- **Multiplication:** $X \cdot Y = iZ$, and the product of two Pauli
  strings is another Pauli string (up to a phase).
- **Commutation:** Two Pauli strings either commute or anticommute.
  They commute iff they anticommute at an even number of qubit positions.
- **Expectation values:** $\langle\psi|P|\psi\rangle$ can be computed in
  $O(2^n)$ time without materializing any matrix.

In [ ]:
%%
xx, _ := pauli.Parse("XX")
yy, _ := pauli.Parse("YY")
zz, _ := pauli.Parse("ZZ")

// Multiplication: XX * YY = ?
product := pauli.Mul(xx, yy)
fmt.Printf("XX * YY = %s\n", product)

// Commutation relations.
fmt.Printf("XX commutes with ZZ: %v\n", pauli.Commutes(xx, zz))
fmt.Printf("XX commutes with YY: %v\n", pauli.Commutes(xx, yy))
fmt.Printf("XY commutes with YX: ")
xy, _ := pauli.Parse("XY")
yx, _ := pauli.Parse("YX")
fmt.Printf("%v\n", pauli.Commutes(xy, yx))

// Expectation values on a Bell state.
sv := statevector.New(2)
bellC, _ := builder.New("bell", 2).H(0).CNOT(0, 1).Build()
sv.Evolve(bellC)

fmt.Printf("\nExpectation values for Bell state |Phi+>:\n")
fmt.Printf("  <Bell|XX|Bell> = %.4f\n", sv.ExpectPauliString(xx))
fmt.Printf("  <Bell|YY|Bell> = %.4f\n", sv.ExpectPauliString(yy))
fmt.Printf("  <Bell|ZZ|Bell> = %.4f\n", sv.ExpectPauliString(zz))

// PauliSum expectation (Hamiltonian).
ham, _ := pauli.NewPauliSum([]pauli.PauliString{
	xx,
	yy.Scale(-0.5),
	zz.Scale(0.3),
})
fmt.Printf("\n  <Bell| XX - 0.5*YY + 0.3*ZZ |Bell> = %.4f\n", sv.ExpectPauliSum(ham))

## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. Why does second-order Trotterization give better accuracy than first-order at the same number of steps?
2. Why can Clifford circuits be simulated efficiently on classical computers, and what gate breaks this efficiency?
3. How does HHL encode the solution to a linear system in quantum amplitudes, and why is post-selection needed?

---

## Exercises

### Exercise 1 -- Trotter Accuracy Comparison

Sweep the number of Trotter steps from 1 to 16 for both first-order
and second-order decompositions of the Heisenberg XXZ Hamiltonian
$H = XX + YY + 0.5 \cdot ZZ$. Plot (or print) the fidelity vs step
count to see how second-order converges faster.

In [ ]:
%%
// Exercise 1: Trotter Accuracy Comparison
//
// Goal: Sweep Trotter steps from 1 to 16 for both first-order and
//       second-order decompositions and compare fidelity vs a reference.
//
// Hamiltonian: H = XX + YY + 0.5*ZZ (Heisenberg XXZ model)
//
// TODO: Build the Hamiltonian
// TODO: Compute a high-precision reference state
// TODO: Sweep step counts and print fidelity for both orders
//
// Skeleton:

ctx := context.Background()

// Step 1: Build the Hamiltonian
// xx, _ := pauli.Parse("XX")
// yy, _ := pauli.Parse("YY")
// zz, _ := pauli.Parse("ZZ")
// ham, _ := pauli.NewPauliSum([]pauli.PauliString{xx, yy, zz.Scale(???)})

// Step 2: Compute a high-precision reference (many steps, second-order)
// svRef, _ := trotter.Evolve(ctx, trotter.Config{
//     Hamiltonian: ???, Time: 1.0, Steps: 200, Order: trotter.Second,
// }, nil)

// Step 3: Define a fidelity function
// fidelity := func(a, b []complex128) float64 {
//     var dot complex128
//     for i := range a {
//         dot += complex(real(a[i]), -imag(a[i])) * b[i]
//     }
//     return real(dot)*real(dot) + imag(dot)*imag(dot)
// }

// Step 4: Sweep step counts and compare
// fmt.Printf("%6s  %16s  %16s\n", "Steps", "1st-order F", "2nd-order F")
// for _, steps := range []int{1, 2, 4, 8, 16} {
//     sv1, _ := trotter.Evolve(ctx, trotter.Config{
//         Hamiltonian: ???, Time: 1.0, Steps: ???, Order: trotter.First,
//     }, nil)
//     sv2, _ := trotter.Evolve(ctx, trotter.Config{
//         Hamiltonian: ???, Time: 1.0, Steps: ???, Order: trotter.Second,
//     }, nil)
//     fmt.Printf("%6d  %16.10f  %16.10f\n", steps, fidelity(svRef, sv1), fidelity(svRef, sv2))
// }

fmt.Println("TODO: Uncomment the code above, fill in the ???, and run!")

## Key Takeaways -- Curriculum Conclusion

Over sixteen notebooks, we have built a comprehensive understanding of
quantum computing using the Goqu SDK:

1. **Hamiltonian simulation** via Trotter-Suzuki decomposition lets us
   approximate $e^{-iHt}$ for arbitrary Hamiltonians. Second-order
   Trotter converges faster than first-order at the same circuit depth.

2. **HHL** solves linear systems $Ax = b$ by encoding the solution in
   quantum amplitudes. It requires QPE, controlled rotations, and
   post-selection.

3. **Clifford simulation** using stabilizer tableaux can handle up to
   100,000 qubits in polynomial time -- but only for Clifford circuits
   (H, S, CNOT, Pauli gates).

4. **Pauli algebra** provides the mathematical foundation for expressing
   Hamiltonians, computing expectation values, and analyzing commutation
   relations -- all in $O(2^n)$ without materializing matrices.

For pulse-level programming, operator representations, QASM/Quil interop,
backends, and observability hooks, see [Notebook 16b](16b-sdk-reference.ipynb).

---

**Congratulations on completing the quantum computing curriculum!**
You now have the tools and understanding to build, simulate, transpile,
and run quantum algorithms -- from single-qubit gates all the way to
production-grade Hamiltonian simulation.

The quantum advantage is not in any single algorithm, but in the ability
to compose all of these pieces: express a problem as a Hamiltonian,
simulate it with the right decomposition, transpile to hardware
constraints, mitigate noise, and monitor the entire pipeline.